# Praktikum 2 - Klasifikasi Iris dengan Keras

**Nama:** Radit Firansah  
**NIM:** 244107020196

Membangun JST menggunakan Keras Sequential untuk mengklasifikasikan dataset Iris.

## Langkah 1 - Import & Load Dataset

Dataset Iris dimuat dari `sklearn.datasets`, lalu label di-*one-hot encoding*.

> **Catatan:** parameter `sparse` pada `OneHotEncoder` sudah diganti menjadi `sparse_output` sejak scikit-learn 1.2.

In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

# Load dataset
iris = load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)

# One-hot encoding
encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

I0000 00:00:1790790301.410378   48172 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790790301.410802   48172 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790790301.443325   48172 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1790790302.411479   48172 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790790302.411909   48172 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Langkah 2 - Bangun Model

Model Sequential dengan dua hidden layer (10 dan 8 neuron, ReLU) dan output 3 kelas softmax.

In [2]:
# Bangun model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu', input_shape=(4,)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

# Kompilasi
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

E0000 00:00:1790790303.015112   48172 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1790790303.015312   48297 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1790790303.031363   48172 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## Langkah 3 - Latih & Evaluasi

In [3]:
# Latih model
model.fit(X_train, y_train, epochs=50, batch_size=8, verbose=0)

# Evaluasi
loss, acc = model.evaluate(X_test, y_test)
print(f"Akurasi: {acc}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.9667 - loss: 0.2103

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.9667 - loss: 0.2103


Akurasi: 0.9666666388511658


## Tugas 2 & 3 - Variasi Neuron & Fungsi Aktivasi

Kita membandingkan beberapa konfigurasi hidden layer dan fungsi aktivasi
(**ReLU** vs **Sigmoid**), lalu mencetak akurasi dan loss tiap konfigurasi.

In [4]:
def buat_model(h1, h2, activation):
    m = tf.keras.Sequential([
        tf.keras.layers.Dense(h1, activation=activation, input_shape=(4,)),
        tf.keras.layers.Dense(h2, activation=activation),
        tf.keras.layers.Dense(3, activation='softmax')
    ])
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return m

konfigurasi = [
    (4, 4, 'relu'),
    (10, 8, 'relu'),   # konfigurasi awal
    (32, 16, 'relu'),
    (10, 8, 'sigmoid'),
]

print(f"{'Hidden':<12}{'Aktivasi':<10}{'Loss':<12}{'Akurasi':<10}")
for h1, h2, act in konfigurasi:
    tf.keras.utils.set_random_seed(42)
    m = buat_model(h1, h2, act)
    m.fit(X_train, y_train, epochs=50, batch_size=8, verbose=0)
    loss, acc = m.evaluate(X_test, y_test, verbose=0)
    print(f"{f'{h1}-{h2}':<12}{act:<10}{loss:<12.4f}{acc:<10.4f}")

Hidden      Aktivasi  Loss        Akurasi   


4-4         relu      0.7143      0.7000    


10-8        relu      0.2103      0.9667    


32-16       relu      0.0924      1.0000    


10-8        sigmoid   0.5842      0.8000    


### Kesimpulan Tugas 2 & 3

- Menambah jumlah neuron hidden umumnya menaikkan akurasi, tetapi setelah titik tertentu peningkatannya tidak signifikan.
- **ReLU** biasanya konvergen lebih cepat dan menghasilkan akurasi lebih tinggi dibanding **Sigmoid** pada dataset ini, karena Sigmoid rentan terhadap *vanishing gradient*.